In [ ]:
import os
import os.path as op
from glob import glob
from typing import Any, Callable, Dict, List, Optional

import pandas as pd

In [ ]:
BASE_IN_DIR = "" # Write your path here
WORKING_DIR = "" # Write your path here
OUT_DIR = os.path.join(WORKING_DIR, "dsets")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
ExistsFn = Callable[[str, str, str], bool]  # (root, subj_path, subj_dir) -> bool
RowFn = Callable[
    [Any, Dict[str, bool]], Dict[str, Any]
]  # (beh_row, flags) -> row extras
KeyFn = Callable[[str], Any]  # subj_dir -> index key for behavior_df
SiteFn = Callable[[str], str]  # root -> site name
FilterFn = Callable[[Any, str], bool]  # (key, root) -> keep?


def collect_dataset(
    root_globs: List[str],
    subject_rel_glob: str,
    behavior_df: Optional[pd.DataFrame],
    key_fn: KeyFn,
    site_fn: SiteFn,
    exists_fns: Dict[str, ExistsFn],
    row_fn: RowFn,
    filter_fn: Optional[FilterFn] = None,
) -> pd.DataFrame:
    """
    Collect dataset rows by scanning subject folders and joining with behavior.

    Parameters
    ----------
    root_globs : list of str
        Glob patterns to dataset roots.
    subject_rel_glob : str
        Relative glob below each root to find subject folders (e.g., 'derivatives/halfpipe/sub-*').
    behavior_df : DataFrame or None
        Behavior table indexed by subject key, or None if not used.
    key_fn : callable
        Maps 'subj_dir' (e.g., 'sub-0001') -> behavior index key.
    site_fn : callable
        Maps 'root' -> site string.
    exists_fns : dict
        Mapping of flag name -> callable(root, subj_path, subj_dir) -> bool.
    row_fn : callable
        Builds extra row fields from behavior row (or None) and flags dict.
    filter_fn : callable or None
        Optional predicate (key, root) -> bool to include/exclude subjects.

    Returns
    -------
    DataFrame
        Assembled rows including 'site', 'subject', any row_fn fields, and flags.
    """
    rows: List[Dict[str, Any]] = []
    roots: List[str] = []
    for pat in root_globs:
        roots.extend(glob(pat))
    for root in sorted(roots):
        site = site_fn(root)
        for subj_path in sorted(glob(op.join(root, subject_rel_glob))):
            subj_dir = op.basename(subj_path)
            flags = {k: f(root, subj_path, subj_dir) for k, f in exists_fns.items()}
            if not any(flags.values()):
                continue
            key = key_fn(subj_dir)
            if behavior_df is not None and key not in behavior_df.index:
                continue
            if filter_fn is not None and not filter_fn(key, root):
                continue
            beh_row = behavior_df.loc[key] if behavior_df is not None else None
            row = {"site": site, "subject": subj_dir}
            row.update(row_fn(beh_row, flags))
            row.update(flags)
            rows.append(row)
    return pd.DataFrame(rows)


def save_and_describe(
    df: pd.DataFrame, out_path: str, age_col: str = "age", sex_col: str = "sex"
) -> None:
    df.to_csv(out_path, index=False)
    if age_col in df.columns:
        print(df[age_col].describe())
    if sex_col in df.columns:
        print(df[sex_col].value_counts())

In [ ]:
# ABCD
abcd_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "abcd/halfpipe/abcd.tsv"), sep="\t"
).set_index("src_subject_id")

abcd_df = collect_dataset(
    root_globs=[op.join(BASE_IN_DIR, "abcd/halfpipe/site*")],
    subject_rel_glob=op.join("derivatives", "halfpipe", "sub-*"),
    behavior_df=abcd_beh,
    key_fn=lambda s: s.replace("sub-", ""),
    site_fn=lambda root: "abcd_" + op.basename(root),
    exists_fns={
        "nback": lambda r, p, s: op.exists(
            op.join(p, "ses-2YearFollowUpYArm1", "func", "task-nback")
        ),
        "mid": lambda r, p, s: op.exists(
            op.join(p, "ses-2YearFollowUpYArm1", "func", "task-mid")
        ),
    },
    row_fn=lambda row, flags: {
        "sex": row["sex"].upper(),
        "age": row["interview_age"] / 12.0,
    },
)
save_and_describe(abcd_df, op.join(OUT_DIR, "abcd_data.csv"))

In [ ]:
# IntegraMOODS
im_beh = pd.read_csv(
    op.join(WORKING_DIR, "data/beh/integramoods.tsv"),
    sep="\t",
)
im_beh["sex"] = im_beh["sex"].replace({"male": "M", "female": "F"})
im_beh = im_beh.set_index("participant_ID")

integrament_df = collect_dataset(
    root_globs=[
        op.join(BASE_IN_DIR, "integramoods/halfpipe/integrament/*"),
        op.join(BASE_IN_DIR, "integramoods/halfpipe/moods/*"),
    ],
    subject_rel_glob=op.join("derivatives", "halfpipe", "sub-*"),
    behavior_df=im_beh,
    key_fn=lambda s: s.replace("sub-", ""),
    site_fn=lambda root: op.basename(op.dirname(root)) + "_" + op.basename(root),
    exists_fns={
        "mid": lambda r, p, s: op.exists(op.join(p, "func", "task-reward")),
        "nback": lambda r, p, s: op.exists(op.join(p, "func", "task-nback")),
    },
    row_fn=lambda row, flags: {
        "group": row["group"],
        "sex": row["sex"],
        "age": row["age"],
    },
    filter_fn=lambda key, root: im_beh.loc[key]["group"] in {"control", "relative"},
)
save_and_describe(integrament_df, op.join(OUT_DIR, "integramoods.csv"))

In [ ]:
# QTIM
qtim_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "queensland/qtim/bids/participants.tsv"), sep="\t"
).set_index("participant_id")

qtim_df = collect_dataset(
    root_globs=[op.join(BASE_IN_DIR, "queensland/qtim/*")],
    subject_rel_glob=op.join("derivatives", "halfpipe", "sub-*"),
    behavior_df=qtim_beh,
    key_fn=lambda s: s,
    site_fn=lambda root: "qtim",
    exists_fns={
        "nback": lambda r, p, s: op.exists(op.join(p, "ses-01", "func", "task-nback")),
    },
    row_fn=lambda row, flags: {
        "family_id": row["family_id"],
        "sex": row["sex"],
        "age": row["age"],
    },
)
save_and_describe(qtim_df, op.join(OUT_DIR, "qtim_data.csv"))
print(qtim_df["family_id"].nunique())

In [ ]:
# DynaMORE
dynamore_beh = pd.read_csv(op.join(WORKING_DIR, "data/beh/dynamore.csv"))
dynamore_beh["subject"] = dynamore_beh["id"].astype(str)
dynamore_beh = dynamore_beh.set_index("subject")

dynamore_df = collect_dataset(
    root_globs=[
        op.join(BASE_IN_DIR, "stressimaging/dynamore/halfpipe/mid/derivatives/halfpipe")
    ],
    subject_rel_glob="sub-*",
    behavior_df=dynamore_beh,
    key_fn=lambda s: s.replace("sub-", ""),
    site_fn=lambda root: None,  # comes from behavior
    exists_fns={"mid": lambda r, p, s: op.exists(op.join(p, "func", "task-mid"))},
    row_fn=lambda row, flags: {
        "site": row["site"],
        "sex": row["gender"],
        "age": row["age"],
    },
)
save_and_describe(dynamore_df, op.join(OUT_DIR, "dynamore_data.csv"))

In [ ]:
# PIOP1
piop1_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "aomic/behavioral/PIOP1_participants.tsv"), sep="\t"
).set_index("participant_id")

piop1_df = collect_dataset(
    root_globs=[op.join(BASE_IN_DIR, "aomic/halfpipe/piop1/derivatives/halfpipe")],
    subject_rel_glob="sub-*",
    behavior_df=piop1_beh,
    key_fn=lambda s: s,
    site_fn=lambda root: "piop1",
    exists_fns={
        "nback": lambda r, p, s: op.exists(op.join(p, "func", "task-workingmemory"))
    },
    row_fn=lambda row, flags: {"sex": row["sex"], "age": row["age"]},
)
save_and_describe(piop1_df, op.join(OUT_DIR, "piop1_data.csv"))

In [ ]:
# PIOP2
piop2_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "aomic/behavioral/PIOP2_participants.tsv"), sep="\t"
).set_index("participant_id")

piop2_df = collect_dataset(
    root_globs=[op.join(BASE_IN_DIR, "aomic/bids/piop2")],
    subject_rel_glob="sub-*",
    behavior_df=piop2_beh,
    key_fn=lambda s: s,
    site_fn=lambda root: "piop2",
    exists_fns={
        "nback": lambda r, p, s: op.exists(
            op.join(p, "func", f"{s}_task-workingmemory_acq-seq_bold.nii.gz")
        ),
    },
    row_fn=lambda row, flags: {"sex": row["sex"], "age": row["age"]},
)
save_and_describe(piop2_df, op.join(OUT_DIR, "piop2_data.csv"))

In [ ]:
# HCP-YA
hcpya_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "hcp/code/lea/young_adult/hcp.tsv"), sep="\t"
).set_index("subject")

hcpya_df = collect_dataset(
    root_globs=[op.join(BASE_IN_DIR, "hcp/halfpipe/young_adult/derivatives/halfpipe")],
    subject_rel_glob="sub-*",
    behavior_df=hcpya_beh,
    key_fn=lambda s: int(s.replace("sub-", "")),
    site_fn=lambda root: "hcp",
    exists_fns={
        "nback": lambda r, p, s: op.exists(op.join(p, "ses-RL", "func", "task-WM")),
        "mid": lambda r, p, s: op.exists(op.join(p, "ses-RL", "func", "task-GAMBLING")),
    },
    row_fn=lambda row, flags: {"sex": row["gender"], "age": row["age"]},
)
save_and_describe(hcpya_df, op.join(OUT_DIR, "hcp_data.csv"))

In [ ]:
# HCP-D
hcpd_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "hcp/code/lea/aging_and_development/spreadsheet.tsv"),
    sep="\t",
).set_index("src_subject_id")

hcpd_df = collect_dataset(
    root_globs=[op.join(BASE_IN_DIR, "hcp/halfpipe/development/derivatives/halfpipe")],
    subject_rel_glob="sub-*",
    behavior_df=hcpd_beh,
    key_fn=lambda s: s.replace("sub-", ""),
    site_fn=lambda root: "hcpd",
    exists_fns={
        "mid": lambda r, p, s: op.exists(op.join(p, "ses-AP", "func", "task-GUESSING"))
    },
    row_fn=lambda row, flags: {"sex": row["sex"], "age": row["interview_age"] / 12.0},
)
save_and_describe(hcpd_df, op.join(OUT_DIR, "hcpd_data.csv"))

In [ ]:
# PNC
pnc_beh = pd.read_csv(op.join(BASE_IN_DIR, "pnc/code/lea/pnc.tsv"), sep="\t").set_index(
    "subject"
)

pnc_df = collect_dataset(
    root_globs=[op.join(BASE_IN_DIR, "pnc/halfpipe/derivatives/halfpipe")],
    subject_rel_glob="sub-*",
    behavior_df=pnc_beh,
    key_fn=lambda s: int(s.replace("sub-", "")),
    site_fn=lambda root: "pnc",
    exists_fns={
        "nback": lambda r, p, s: op.exists(op.join(p, "func", "task-frac2back"))
    },
    row_fn=lambda row, flags: {"sex": row["sex"], "age": row["age"] / 12.0},
)
save_and_describe(pnc_df, op.join(OUT_DIR, "pnc_data.csv"))

In [ ]:
# WAHN
wahn_beh = pd.read_csv(op.join(BASE_IN_DIR, "wahn/behavioral/WAHN_Daten.csv")).rename(
    {"ID": "participant_ID", "Alter": "age"}, axis=1
)[["participant_ID", "age", "Geschlecht"]]
wahn_beh["participant_ID"] = wahn_beh["participant_ID"].str.replace("WAHN_", "sub-")
wahn_beh["Geschlecht"] = wahn_beh["Geschlecht"].replace({"1": "F", "2": "M"})
wahn_beh = wahn_beh.set_index("participant_ID")

wahn_df = collect_dataset(
    root_globs=[op.join(BASE_IN_DIR, "wahn/halfpipe/nback/derivatives/halfpipe")],
    subject_rel_glob="sub-*",
    behavior_df=wahn_beh,
    key_fn=lambda s: s,
    site_fn=lambda root: "wahn",
    exists_fns={
        "nback": lambda r, p, s: op.exists(
            op.join(p, "func", f"{s}_task-nback_setting-aroma_desc-brain_mask.nii.gz")
        ),
    },
    row_fn=lambda row, flags: {"sex": row["Geschlecht"], "age": row["age"]},
)
save_and_describe(wahn_df, op.join(OUT_DIR, "wahn_data.csv"))

In [ ]:
# IMAGEN (new CSV; long output with columns: age, fu)
# IMAGEN
imagen_df = pd.read_csv(
    op.join(WORKING_DIR, "data/beh/imagen.csv"),
    converters={"subject": str},
).drop(columns=["handedness"])
imagen_df["site"] = [f"imagen_{s}" for s in imagen_df["site"]]
imagen_df["subject"] = imagen_df["subject"].str.zfill(12)
imagen_df["sex"] = imagen_df["sex"].str.upper()
imagen_df

save_and_describe(imagen_df, op.join(OUT_DIR, "imagen_data.csv"))
imagen_df[["age", "fu"]].groupby("fu").mean()

In [ ]:
# CHCP
chcp_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "hcp/raw/chinese/nifti/CHCP_subjects_information.csv")
).set_index("Subject")

chcp_df = collect_dataset(
    root_globs=[op.join(BASE_IN_DIR, "hcp/raw/chinese/preproc")],
    subject_rel_glob="*",
    behavior_df=chcp_beh,
    key_fn=lambda s: int(s),
    site_fn=lambda root: "chcp",
    exists_fns={
        "nback": lambda r, p, s: op.exists(
            op.join(
                p,
                "MNINonLinear",
                "Results",
                "tfMRI_Nback",
                "tfMRI_Nback_hp200_s2_level2.feat",
                "StandardVolumeStats",
            )
        ),
        "mid": lambda r, p, s: op.exists(
            op.join(
                p,
                "MNINonLinear",
                "Results",
                "tfMRI_Gambling",
                "tfMRI_Gambling_hp200_s2_level2.feat",
                "StandardVolumeStats",
            )
        ),
    },
    row_fn=lambda row, flags: {"age": row["Age_in_Yrs"], "sex": row["Gender"]},
)
chcp_df["sex"] = [sex[0] for sex in chcp_df["sex"]]
save_and_describe(chcp_df, op.join(OUT_DIR, "chcp_data.csv"))

In [ ]:
# ds003849
ds003849_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "openneuro/ds003849/behavioral/plasticity_mri_sample.csv"),
).rename(columns={"record_id": "subject"})[["subject", "age", "sex"]]
ds003849_beh["nback"] = True
ds003849_beh["site"] = "ds003849"
ds003849_beh.to_csv(op.join(OUT_DIR, "ds003849_data.csv"), index=False)


In [ ]:
# ds003858
ds003858_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "openneuro/ds003858/bids/participants.tsv"),
    sep="\t",
).rename(columns={"gender": "sex"})
ds003858_beh["mid"] = True
ds003858_beh["site"] = "ds003858"
ds003858_beh.to_csv(op.join(OUT_DIR, "ds003858_data.csv"), index=False)


In [ ]:
# ds005479
# All the subjects are male!
ds005479_beh = pd.read_csv(
    op.join(BASE_IN_DIR, "openneuro/ds005479/bids/participants.tsv"),
    sep="\t",
)
ds005479_beh["sex"] = "M"
ds005479_beh["mid"] = True
ds005479_beh["site"] = "ds005479"
ds005479_beh.to_csv(op.join(OUT_DIR, "ds005479_data.csv"), index=False)
